<a href="https://colab.research.google.com/github/olawoye/nla_phd/blob/main/notebooks/setup_activations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Section 1: SOme Basic Setup**

In [ ]:
# 1.1 INSTALL LIBRARIES & SETUP GOOGLE DRIVE (for storage/backup)

# Lets instal basic libraries
!pip install -q transformers
!pip install -q accelerate
!pip install -q datasets
!pip install -q pyarrow
!pip install -q pandas
!pip install -q sentencepiece

# For Gemma
!pip install -q torch

# fix issuess saving to my github
!pip install ipywidgets

import json
import os
import torch
import pandas as pd
import platform
import glob
from google.colab import userdata
import shutil

import os
import shutil
import glob
import pandas as pd
from google.colab import userdata

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
import numpy as np

import torch.nn as nn
import torch.nn.functional as F

# Setup my project folder and additional configuration
from google.colab import drive
drive.mount('/content/drive')

USE_GOOGLE_DRIVE = False  # Set to True to save to Drive, False to keep local

from google.colab import userdata

# Securely retrieve the token from Colab's secret vault
github_token = userdata.get('GITHUB_TOKEN')

# Inject it into the system environment variables for Git to use
os.environ['GITHUB_TOKEN'] = github_token

# Run Parameters
RESUME_MODE = False
MODEL_NAME = "google/gemma-2-2b"

# Set other parameters
NUM_SAMPLES_TO_PROCESS = 5000  # I can easily set to 10, 100, 10000, or 50000
CHECKPOINT_EVERY = 50        # Save data periodically to prevent loss
TARGET_LAYER = 8             # The internal transformer layer to extract according to documen
TEXT_TRUNCATION_LIMIT = 1000 # Character limit per document to keep generation fast

# Project parameters
GH_USER = "olawoye"
REPO_NAME = "nla_phd"
LOCAL_REPO_PATH = f"/content/{REPO_NAME}"
TARGET_LAYER = 8
REPO_DOMAIN = "github.com"
PROJECT_DIR = f"/content/drive/MyDrive/{REPO_NAME}"

# Experiment directory
CHECKPOINT_PATH = f"{NUM_SAMPLES_TO_PROCESS}_samples"

RUN_DIR = os.path.join(
    PROJECT_DIR,
    CHECKPOINT_PATH
)

os.makedirs(RUN_DIR, exist_ok=True)
OUTPUT_FILENAME = os.path.join(
    RUN_DIR,
    "gemma_activations_final.parquet"
)

# Ensure all experiment directories exist
RUN_DIR = os.path.join(
    PROJECT_DIR,
    CHECKPOINT_PATH
)

os.makedirs(RUN_DIR, exist_ok=True)

ACTIVATION_CHECKPOINT_DIR = os.path.join(
    RUN_DIR,
    "activation_checkpoints"
)

CHECKPOINT_DIR = os.path.join(
    RUN_DIR,
    "checkpoints"
)

VERBALIZER_CHECKPOINTS_DIR = os.path.join(
    RUN_DIR,
    "verbalizer_checkpoint"
)

VERBALIZER_TOKENIZER_DIR = os.path.join(
    RUN_DIR,
    "verbalizer_tokenizer"
)

for d in [
    ACTIVATION_CHECKPOINT_DIR,
    CHECKPOINT_DIR,
    VERBALIZER_CHECKPOINTS_DIR,
    VERBALIZER_TOKENIZER_DIR
]:
    os.makedirs(d, exist_ok=True)

# --- HARD RESET SYSTEM DIRECTORY PATH ---
# Forces the entire notebook engine out of any bad background directory state
os.chdir("/content")

# --- 2. SECURE CLONE OR UPDATE ---
is_valid_git_repo = os.path.exists(LOCAL_REPO_PATH) and os.path.exists(os.path.join(LOCAL_REPO_PATH, ".git"))

if not is_valid_git_repo:
    if os.path.exists(LOCAL_REPO_PATH):
        print("Cleaning up partial directory structure...")
        shutil.rmtree(LOCAL_REPO_PATH)

    print("Fresh Colab session detected. Cloning repository cleanly...")
    try:
        gh_token = userdata.get('GITHUB_TOKEN')

        # FIXED STRING STRUCTURE: Explicitly split elements to block URL mangling bugs
        domain = REPO_DOMAIN
        repo_url = f"https://{gh_token}@{domain}/{GH_USER}/{REPO_NAME}.git"

        # Run clone using precise destination tracking
        !git clone "{repo_url}" "{LOCAL_REPO_PATH}"

    except Exception as e:
        print(f"Error accessing GitHub token secret: {e}")
        print("Setting up local folders as an offline fallback...")
        os.makedirs(LOCAL_REPO_PATH, exist_ok=True)
else:
    print(" Repository exists and is valid. Pulling latest changes...")
    os.chdir(LOCAL_REPO_PATH)
    !git pull origin main

# Always drop back to system root after git tasks
os.chdir("/content")

# --- 3. ENSURE DIRECTORY STRUCTURE ---
os.makedirs(os.path.join(LOCAL_REPO_PATH, "data"), exist_ok=True)
os.makedirs(os.path.join(LOCAL_REPO_PATH, f"data/{CHECKPOINT_EVERY}"), exist_ok=True)
os.makedirs(os.path.join(LOCAL_REPO_PATH, "notebooks"), exist_ok=True)

# --- 4. SAFE .GITIGNORE CREATION ---
gitignore_content = """
data/large_combinations.parquet
data/*_50k.parquet
.ipynb_checkpoints/
"""
gitignore_path = os.path.join(LOCAL_REPO_PATH, ".gitignore")
with open(gitignore_path, "w") as f:
    f.write(gitignore_content.strip())
print("✓ .gitignore verified and updated.")

# --- 5. AUTOMATED PROGRESS RECOVERY ---
# def load_existing_progress():
#     data_dir = os.path.join(LOCAL_REPO_PATH, "data")
#     parquet_files = glob.glob(os.path.join(data_dir, "checkpoint_*.parquet"))

#     if not parquet_files:
#         print("\n No previous saved checkpoints found in our repo. Starting fresh (Index: 0).")
#         return pd.DataFrame(), 0

#     print(f"\n Found {len(parquet_files)} saved checkpoint files. Reloading into memory...")
#     df_list = [pd.read_parquet(f) for f in parquet_files]
#     combined_df = pd.concat(df_list, ignore_index=True)

#     next_index = int(combined_df["sample_index"].max() + 1) if "sample_index" in combined_df.columns else 0
#     print(f"✓ Successfully reloaded {len(combined_df)} historical activations.")
#     print(f" Reset complete. Ready to run starting at sample index: {next_index}")
#     return combined_df, next_index
def load_existing_progress():

  print("Checkpoint directory:", ACTIVATION_CHECKPOINT_DIR)

  if not RESUME_MODE:
      print("\nRESUME_MODE=False. Starting fresh.")
      return pd.DataFrame(), 0

  parquet_files = glob.glob(
      os.path.join(
          ACTIVATION_CHECKPOINT_DIR,
          "checkpoint_*.parquet"
      )
  )

  if not parquet_files:
      print(
          f"\nNo checkpoints found in "
          f"{ACTIVATION_CHECKPOINT_DIR}"
      )
      return pd.DataFrame(), 0

  print(
      f"\nFound {len(parquet_files)} checkpoint files "
      f"in {ACTIVATION_CHECKPOINT_DIR}"
  )

  df_list = [pd.read_parquet(f) for f in parquet_files]
  combined_df = pd.concat(df_list, ignore_index=True)

  next_index = (
      int(combined_df["sample_index"].max() + 1)
      if "sample_index" in combined_df.columns
      else 0
  )

  print(
      f"Reloaded {len(combined_df)} activations. "
      f"Resuming from sample {next_index}"
  )

  return combined_df, next_index

# Run the recovery function automatically
historical_data, starting_sample_index = load_existing_progress()


PIP_FREEZE_PATH = os.path.join(
    RUN_DIR,
    "requirements.txt"
)

# Save PIP Requirements
#!pip freeze > "{PROJECT_DIR}/requirements.txt"
!pip freeze > "{PIP_FREEZE_PATH}"

# Restore
#!head "{PROJECT_DIR}/requirements.txt"
!head "{PIP_FREEZE_PATH}"







In [ ]:
# 1.2 SAVE META DATA and OTHER REQUIREMENTS (for future reproducibility

experiment_info = {

    "model_name": MODEL_NAME,

    "activation_layer": TARGET_LAYER,

    #"embedding_model": "all-MiniLM-L6-v2",
    "embedding_model": "google/gemma-2-2b",

    "python_version": platform.python_version(),

    "torch_version": torch.__version__,

    "dataset": "OpenWebText"
}

with open(
    os.path.join(
        RUN_DIR,
        "experiment_info.json"
    ),
    "w"
) as f:

    json.dump(
        experiment_info,
        f,
        indent=2
    )

# **Section 2: Model Setup**

In [ ]:
# 2.1 HUGGING FACE LOGIN (and necessary approval of datasets as we go alon)

from huggingface_hub import login

login()

In [ ]:
# 2.2 GEMMA-2-2B and Tokenizer

# So I'm kind of experiementing with either Gemma-2-2B and Qwen2.5-Coder-3B (because its seems with better code inclined)
# but Gemma seems to better SAE ecosytem to leverage faster

# Gemma Option
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

model_name = MODEL_NAME

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

# Should we decide on Qwen
'''
model_name = "Qwen/Qwen2.5-Coder-3B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)
'''

# Lets see
print(model)

In [ ]:
# 2.3 SINGLE TEST (Lets run a simple inference from the model just to get an idea)


if USE_GOOGLE_DRIVE:
    save_dir = PROJECT_DIR + "/colab_outputs"
else:
    save_dir = "/content/local_colab_outputs"

os.makedirs(save_dir, exist_ok=True)

# --- prompts for maybe later BULK RUN ---
prompts_list = [
    "Write a Python function that computes factorial.",
    "Write a Python function that checks prime numbers.",
    # Add thousands of prompts here...
]
# eventually used a code dataset so ignore

# --- manually define the Gemma chat template (our base model doesnt carry this)
if tokenizer.chat_template is None:
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "{{'<start_of_turn>' + message['role'] + '\n' + message['content'] + '<end_of_turn>\n'}}"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
        "{{'<start_of_turn>assistant\n'}}"
        "{% endif %}"
    )

# --- BULK RUN LOOP ---
for idx, prompt_text in enumerate(prompts_list):
    chat = [{"role": "user", "content": f"{prompt_text} Output only the code."}]
    prompt = tokenizer.apply_chat_template(
        chat, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Inference
    output = model.generate(
        **inputs,
        max_new_tokens=100,
        output_hidden_states=True,
        return_dict_in_generate=True,
    )


    # --- SHAPE INSPECTION PRINTING BLOCK ---

    # 1. output.hidden_states is a tuple where each element is a generation step
    num_steps = len(output.hidden_states)

    print("Shape:\n")

    # Let's inspect the very first step (Step 0) of the generation
    first_step_layers = output.hidden_states[0]

    # Extract Layer 8 (index 8) from this step
    # This gives you the full tensor before any slicing or memory optimization
    layer_8 = first_step_layers[8]

    # Print the code and dynamic shape dynamically
    print("print(layer_8.shape)\n")
    print(f"{tuple(layer_8.shape)}\n")

    # 2. Extract dimensions from the shape tensor
    batch_size, sequence_length, hidden_dimensions = layer_8.shape

    print("Meaning:\n")
    print(f"batch size = {batch_size}\n")
    print(f"{sequence_length} tokens\n")
    print(f"{hidden_dimensions} hidden dimensions\n")
    print("-" * 40)



    # Text extraction
    input_length = inputs.input_ids.shape[1]
    generated_tokens = output.sequences[0, input_length:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    # Layer 8 Extraction (Memory Optimized)
    layer_8_activations = []
    for step in range(len(output.hidden_states)):
        # Grab layer 8, extract last token slice, drop to CPU list immediately
        last_token_act = output.hidden_states[step][8][0, -1, :]
        layer_8_activations.append(last_token_act.detach().cpu().tolist())

    # Build individual save payload
    output_data = {
        "prompt": prompt_text,
        "generated_code": generated_text,
        "layer_8_activations": layer_8_activations,
    }

    # Save incrementally to prevent data loss if Colab disconnects
    file_path = os.path.join(save_dir, f"result_{idx}.json")
    with open(file_path, "w") as f:
        json.dump(output_data, f)

print(f"Bulk run completed! Files saved in: {save_dir}")


# Section 3: Activation Extraction with Real Data

In [ ]:
# 3.1 LOAD STREAMING OPENWEBTEXT DATA

# Lets get ready to do this for larger data

print(f"COnfiguring to to process {NUM_SAMPLES_TO_PROCESS} samples with checkpoints every {CHECKPOINT_EVERY} steps.")

# --- LOAD DATASET ---
from datasets import load_dataset

# Downloading/Loading OpenWebText dataset from Hugging Face (we use streaming=True download all 40GB of OpenWebText to Colab
dataset = load_dataset("Skylion007/openwebtext", split="train", streaming=True)
dataset_iter = iter(dataset)

# Test print the first item sample snippet to see it works
test_item = next(dataset_iter)
print(test_item["text"][:200] + "...")


In [ ]:
# 3.2 - DEFINE EXTRACTION FUNCTION (extract_layer_activation)
# Put layer activation in a single func to re-use easily and keep tweaks in one plac

def extract_layer_activation(text_input, layer_index=8):
    """
    Passes text through the model and extracts the hidden state activation
    vector of the final token at a specific layer.
    """
    # Truncate text to keep sequence lengths uniform and fast
    truncated_text = text_input[:TEXT_TRUNCATION_LIMIT]

    inputs = tokenizer(
        truncated_text,
        return_tensors="pt",
        truncation=True
    ).to(model.device)

    # We use torch.no_grad() because we are NOT training the LLM; this saves massive memory
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # Shape: [1, seq_len, 2304]
    full_layer_tensor = outputs.hidden_states[layer_index]

    # Slicing out [0, -1] yields the final token vector -> Shape: [2304]
    final_token_vector = full_layer_tensor[0, -1, :]

    # CRITICAL STEP FOR 50K RUNS: Move to CPU and cast to numpy to free up GPU cache
    activation_array = final_token_vector.detach().cpu().numpy()

    return activation_array, truncated_text


In [ ]:
# 3.3 BULK LOOP (with checkpoints)
# and so we start data collection...

# Extract the official model identifier string from loaded model architecture
MODEL_NAME_IDENTIFIER = getattr(model.config, "_name_or_path", "google/gemma-2-2b")

# Calculate upper iteration bound
target_stop_index = starting_sample_index + NUM_SAMPLES_TO_PROCESS

print(f"Resuming run from index {starting_sample_index} using model: {MODEL_NAME_IDENTIFIER}")
print(f"Targeting layer: {TARGET_LAYER} | Upper limit: {target_stop_index}")

# Container to accumulate our metadata-rich records for this session
records = []

# --- THE METADATA-RICH EXTRACTION LOOP ---
for i in range(target_stop_index):
    try:
        document = next(dataset_iter)

        # Skip samples already calculated and committed in historical runs
        if i < starting_sample_index:
            continue

        raw_text = document["text"]

        # 1. Run inference and extract activations
        # We also need the token lengths to determine the exact position of the final token
        truncated_text = raw_text[:TEXT_TRUNCATION_LIMIT]
        inputs = tokenizer(truncated_text, return_tensors="pt", truncation=True).to(model.device)

        # Calculate the absolute position index of the final token (-1 in Python indexing)
        # For a prompt with 18 tokens, inputs.input_ids.shape[1] is 18, so the final token index is 17
        final_token_position = int(inputs.input_ids.shape[1] - 1)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        # Extract the vector slice at Layer 8, Final Token position
        final_token_vector = outputs.hidden_states[TARGET_LAYER][0, -1, :]
        activation_array = final_token_vector.detach().cpu().numpy().tolist()

        # 2. Build the Anthropic NLA Recommended Dataset Structure
        # We attach structural metadata alongside the numeric array
        records.append({
            "sample_index": i,
            "model_name": MODEL_NAME_IDENTIFIER,       # Tracks exactly which model architecture generated this
            "layer_number": int(TARGET_LAYER),         # Confirms target layer index explicitly
            "token_position": final_token_position,    # Tracks absolute position of the token in the context window
            "input_text": truncated_text,              # Stores source prompt context
            "activation_vector": activation_array      # Dense floating point array (Dimension: 2304)
        })

    except StopIteration:
        print("Reached the absolute end of the OpenWebText data stream.")
        break
    except Exception as e:
        print(f"Skipped sample entry at index {i} due to an error: {e}")
        continue

    # 3. Dynamic Checkpoint Exporter
    if (i + 1) % CHECKPOINT_EVERY == 0:
        checkpoint_df = pd.DataFrame(records)

        #checkpoint_path = os.path.join(LOCAL_REPO_PATH, f"data/{CHECKPOINT_PATH}/checkpoint_step_{i + 1}.parquet")
        checkpoint_path = os.path.join(
            RUN_DIR,
            "activation_checkpoints",
            f"checkpoint_step_{i+1}.parquet"
        )
        checkpoint_df.to_parquet(checkpoint_path)
        print(f"✓ Session Checkpoint Exported: {checkpoint_path} (Current Rows: {len(records)})")

print(f"\nProcessing Complete. Appended {len(records)} metadata-enriched rows to the activation pool.")




In [ ]:
# 3.4 - Add everything into a final DataFrame (and .paraquet output format)

final_df = pd.DataFrame(records)

# Save to file
final_df.to_parquet(OUTPUT_FILENAME)
print(f"final dataset successfully created: {OUTPUT_FILENAME}")

# Lets verify
print("\n--- DATASET VERIFICATION ---")
print(f"Total Rows: {final_df.shape[0]}")
print(f"Columns: {list(final_df.columns)}")

# Check the vector length of the first row's activation column
vector_length = len(final_df["activation_vector"].iloc[0])
print(f"Extracted Vector Hidden Dimension Size: {vector_length} (Expected: 2304)")

# Show final layout preview
final_df.head(2)


# **Section 4: Baseline Reconstructorn / Training**

In [ ]:
# 4.1 Setup Transformers, paraquet & Semantic summary targets method

!pip install sentence-transformers

#df = pd.read_parquet(os.path.join("/content", OUTPUT_FILENAME))
df = pd.read_parquet(OUTPUT_FILENAME)

print(df.shape)

df.head()

# SEMANTIC SUMMARY TARGETS (gives the verbalizer a learnable, interpretable supervision signal)

def make_summary_target(text: str, max_chars: int = 200) -> str:
    """
    Produce a short topic-sentence summary without a large LLM.
    Strategy: take the first complete sentence of the snippet (up to 200 chars)
    and prepend a lightweight topic header extracted from the first noun phrase.
    This is a reasonable proxy for the Claude-generated summaries in the paper.
    """
    snippet = text[:500].strip()
    # Find first sentence boundary
    for sep in ['. ', '! ', '? ', '\n']:
        idx = snippet.find(sep)
        if 20 < idx < 250:
            first_sent = snippet[:idx + 1].strip()
            break
    else:
        first_sent = snippet[:200].strip()

    # Truncate to max_chars
    return first_sent[:max_chars]

# Build summary targets and attach to the dataframe
df["summary_target"] = df["input_text"].apply(make_summary_target)

# Quick sanity check
for i in range(3):
    print(f"[{i}] {df['summary_target'].iloc[i]}\n")

In [ ]:
# 4.2 Define Gemma-based text encoder (replaces MiniLM) & Generate embeddings

def encode_text_with_gemma(texts, batch_size=16, layer_index=TARGET_LAYER):
    """
    Encodes a list of texts using Gemma layer-8 final-token vectors.
    Returns numpy array of shape (N, 2304).
    This is the same operation as extract_layer_activation but batched.
    """
    all_vectors = []
    model.eval()

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i: i + batch_size]

        for text in batch_texts:
            truncated = text[:TEXT_TRUNCATION_LIMIT]
            inputs = tokenizer(
                truncated,
                return_tensors="pt",
                truncation=True
            ).to(model.device)

            with torch.no_grad():
                outputs = model(**inputs, output_hidden_states=True)

            vec = outputs.hidden_states[layer_index][0, -1, :]
            all_vectors.append(vec.detach().cpu().numpy())

        if (i + batch_size) % 100 == 0:
            print(f"Encoded {min(i + batch_size, len(texts))}/{len(texts)}...")

    return np.array(all_vectors)  # shape: (N, 2304)

print("Gemma encoder function defined.")


# Generate embeddings

'''
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

texts = df["input_text"].tolist()

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    batch_size=32
)

print(embeddings.shape)
'''
# Now Gemma is better to leverage most of our activation data withoug throwing it away

texts = df["input_text"].tolist()

print(f"Encoding {len(texts)} texts with Gemma layer {TARGET_LAYER}...")
embeddings = encode_text_with_gemma(texts)

print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
# 4.3 - Augment training distribution with short summary-length texts
# This prevents the reconstructor from collapsing on short verbalizer outputs

# Prepare Activator Targets
y = np.array(df["activation_vector"].tolist())

print(y.shape)

summary_texts = df["summary_target"].tolist()
print("Encoding summary targets with Gemma for reconstructor augmentation...")
summary_embeddings = encode_text_with_gemma(summary_texts)

# Stack original and summary embeddings with their shared activation targets
# Each input text and its summary both map to the same activation vector
embeddings_augmented = np.vstack([embeddings, summary_embeddings])
y_augmented = np.vstack([y, y])  # same targets repeated

print(f"Augmented embeddings shape: {embeddings_augmented.shape}")
print(f"Augmented targets shape:    {y_augmented.shape}")

# Use augmented data for all downstream training
embeddings = embeddings_augmented
y = y_augmented


In [ ]:
# 4.4 Define ACtivation Reconstructor

import torch
import torch.nn as nn

class ActivationReconstructor(nn.Module):

    def __init__(self,
                 input_dim,
                 activation_dim):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim, 1024),
            nn.ReLU(),

            nn.Linear(1024, 2048),
            nn.ReLU(),

            nn.Linear(2048, activation_dim)
        )

    def forward(self, x):
        return self.network(x)

#instantiate it
model_reconstructor = ActivationReconstructor(
    #input_dim=384,
    input_dim=embeddings.shape[1],  # now 2304, auto-adapts to whatever encoder used
    activation_dim=y.shape[1]
)

## Lets Begin Actual Training

In [ ]:
# 4.5: Lets create Train/Test split

X_train, X_test, y_train, y_test = train_test_split(
    embeddings,
    y,
    test_size=0.2,
    random_state=42
)

# Convert to tensors
X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
)

# Implement Training
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model_reconstructor.parameters(),
    lr=1e-4
)

In [ ]:
# 4.6: Mechanism to save/ resume progress

# CHECKPOINT_DIR = f"{PROJECT_DIR}/{CHECKPOINT_PATH}/checkpoints"
CHECKPOINT_DIR = os.path.join(
    RUN_DIR,
    "checkpoints"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)


start_epoch = 0

if RESUME_MODE:

    checkpoints = sorted(
        glob.glob(
            f"{CHECKPOINT_DIR}/checkpoint_epoch_*.pt"
        )
    )

    if len(checkpoints) > 0:

        latest_checkpoint = checkpoints[-1]

        checkpoint = torch.load(
            latest_checkpoint,
            map_location="cpu"
        )

        model_reconstructor.load_state_dict(
            checkpoint["model_state_dict"]
        )

        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

        start_epoch = checkpoint["epoch"]

        print(
            f"Resumed from: {latest_checkpoint}"
        )

    else:

        print(
            "RESUME_MODE=True but no checkpoint found."
        )

else:

    print("Starting fresh training")

In [ ]:
# 4.7 Tuning and Saving Epochs

epochs = 20

#for epoch in range(epochs):
for epoch in range(start_epoch, epochs):

    optimizer.zero_grad()

    predictions = model_reconstructor(
        X_train
    )

    loss = criterion(
        predictions,
        y_train
    )

    loss.backward()

    optimizer.step()

    print(
        f"Epoch {epoch+1}: "
        f"{loss.item():.6f}"
    )

    # Save every 5 epochs

    if (epoch + 1) % 5 == 0:

        checkpoint_path = (
            f"{CHECKPOINT_DIR}/"
            f"checkpoint_epoch_{epoch+1}.pt"
        )

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict":
                    model_reconstructor.state_dict(),
                "optimizer_state_dict":
                    optimizer.state_dict(),
                "loss":
                    loss.item()
            },
            checkpoint_path
        )

        print(
            f"Checkpoint saved:"
            f" {checkpoint_path}"
        )

In [ ]:
# 4.8 Cosine Similarity Evaluation
import torch.nn.functional as F

with torch.no_grad():

    predictions = model_reconstructor(
        X_test
    )

    cosine_scores = F.cosine_similarity(
        predictions,
        y_test,
        dim=1
    )

print(
    cosine_scores.mean().item()
)

# 0.0 = random
# 0.2 = weak
# 0.4 = decent
# 0.6 = strong
# 0.8+ = excellent

In [ ]:
# 4.9 FVE Evaluation Checks

with torch.no_grad():

    predictions = model_reconstructor(
        X_test
    )

pred = predictions.numpy()
true = y_test.numpy()

numerator = np.var(
    true - pred
)

denominator = np.var(
    true
)

fve = 1 - (numerator / denominator)

print(
    f"FVE: {fve:.4f}"
)

# 0.00 = random
# 0.10 = weak
# 0.20 = interesting
# 0.40 = good
# 0.60+ = excellent

In [ ]:
# Introspection

# when we have Cosine similarity > 0.2 and FVE > 0.1 is it ideal to proceed

In [ ]:
# 4.10 Baseline A/B Model Experiment


# -------------------------
# Baseline A: Linear
# -------------------------

linear_model = nn.Linear(
    X_train.shape[1],
    y_train.shape[1]
)

optimizer = torch.optim.Adam(
    linear_model.parameters(),
    lr=1e-3
)

for epoch in range(20):

    optimizer.zero_grad()

    pred = linear_model(X_train)

    loss = F.mse_loss(pred, y_train)

    loss.backward()

    optimizer.step()

# Evaluation
with torch.no_grad():

    pred_linear = linear_model(X_test)

linear_cos = F.cosine_similarity(
    pred_linear,
    y_test,
    dim=1
).mean().item()

linear_fve = 1 - (
    np.var(
        y_test.numpy() - pred_linear.numpy()
    )
    /
    np.var(
        y_test.numpy()
    )
)

# -------------------------
# Baseline B: MLP
# -------------------------

mlp_model = nn.Sequential(
    nn.Linear(X_train.shape[1], 1024),
    nn.ReLU(),
    nn.Linear(1024, y_train.shape[1])
)

optimizer = torch.optim.Adam(
    mlp_model.parameters(),
    lr=1e-4
)

for epoch in range(20):

    optimizer.zero_grad()

    pred = mlp_model(X_train)

    loss = F.mse_loss(pred, y_train)

    loss.backward()

    optimizer.step()

# Evaluation
with torch.no_grad():

    pred_mlp = mlp_model(X_test)

mlp_cos = F.cosine_similarity(
    pred_mlp,
    y_test,
    dim=1
).mean().item()

mlp_fve = 1 - (
    np.var(
        y_test.numpy() - pred_mlp.numpy()
    )
    /
    np.var(
        y_test.numpy()
    )
)

# -------------------------
# Results
# -------------------------

print("\n===== RESULTS =====")

print(
    f"Linear Cosine Similarity: {linear_cos:.4f}"
)

print(
    f"Linear FVE: {linear_fve:.4f}"
)

print(
    f"MLP Cosine Similarity: {mlp_cos:.4f}"
)

print(
    f"MLP FVE: {mlp_fve:.4f}"
)

# Bad
'''
Linear Cos = 0.02
Linear FVE = 0.01

MLP Cos = 0.03
MLP FVE = 0.01

Likely: dataset too small; wrong layer; activations too noisy
'''

# Encouraging

'''
Linear Cos = 0.18
Linear FVE = 0.09

MLP Cos = 0.31
MLP FVE = 0.17

This suggests the activation contains recoverable semantic information.
'''

# Very Good
'''
Linear Cos = 0.32
Linear FVE = 0.19

MLP Cos = 0.48
MLP FVE = 0.31

This would strongly justify proceeding.
'''

In [ ]:
# 4.11 Auto propose and auto-save the better model

candidates = {
    "linear": {
        "model": linear_model,
        "cos": linear_cos,
        "fve": linear_fve
    },
    "mlp": {
        "model": mlp_model,
        "cos": mlp_cos,
        "fve": mlp_fve
    }
}

# Score each candidate — normalise both metrics equally
# A combined score rewards models that are strong on both axes
def combined_score(cos, fve):
    return (cos + fve) / 2

best_name = max(
    candidates,
    key=lambda k: combined_score(
        candidates[k]["cos"],
        candidates[k]["fve"]
    )
)

best = candidates[best_name]

print("===== AUTO-SELECTION =====")
for name, c in candidates.items():
    marker = " <-- SELECTED" if name == best_name else ""
    print(
        f"{name.upper():8s} | cos={c['cos']:.4f} | fve={c['fve']:.4f} "
        f"| score={combined_score(c['cos'], c['fve']):.4f}{marker}"
    )

# torch.save(best["model"].state_dict(), "baseline_reconstructor.pt")

BASELINE_RECONSTRUCTOR_PATH = os.path.join(
    RUN_DIR,
    "baseline_reconstructor.pt"
)

torch.save(
    best["model"].state_dict(),
    BASELINE_RECONSTRUCTOR_PATH
)


print(f"\nSaved '{best_name}' model as baseline_reconstructor.pt")

In [ ]:
# 4.12 LINEAR & MLP METRICS CHECK (We need to know if the activations so far are decent before proceeding with verbalizer)

computed_linear_cos = linear_cos
computed_linear_fve = linear_fve
computed_mlp_cos = mlp_cos
computed_mlp_fve = mlp_fve

print("--- EVALUATING MODEL ACTIVATIONS ---")
print(f"Linear Cosine Similarity: {computed_linear_cos:.4f} | Linear FVE: {computed_linear_fve:.4f}")
print(f"MLP Cosine Similarity:    {computed_mlp_cos:.4f} | MLP FVE:    {computed_mlp_fve:.4f}\n")


# We define the upper boundary for "Bad" metrics
if ( (computed_linear_cos >= 0.32 and computed_linear_fve >= 0.19) or
      (computed_mlp_cos >= 0.48 and computed_mlp_fve >= 0.31)):

    print("TIER: VERY GOOD")
    print("Success! This strongly justifies proceeding with the full Natural Language Autoencoder training.")

else:

    # Throwing an error stops all remaining cells
    '''
    raise RuntimeError(
        "\n"
        "CRITICAL STOP: Activation quality is too low to justify proceeding.\n"
        "Please fix data loop or extraction layer before running downstream tasks."
    )
    '''

print("\nSafe to continue execution.")


# **Section 5: ACtivation Verbalizer (AV)**

In [ ]:
# 5.1 Setup / Compress 2304 via PCA

activations = np.array(
    df["activation_vector"].tolist()
)

pca = PCA(
    n_components=64
)

activation_latents = pca.fit_transform(
    activations
)

print(
    activation_latents.shape
)

In [ ]:
# 5.2 - The summary_target bas been relocated to Cell 4.1,
# so the dataframe reference in 5.5 (verbalizer_df) still works correctly.

## **Building Verbalizer**

In [ ]:
# 5.3: Setup Verbalizer Decoder (using google flan-t5)

print(df.columns)
print(df.head())

!pip install -q transformers datasets

from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

VERBALIZER_MODEL = "google/flan-t5-small"

verbalizer_tokenizer = AutoTokenizer.from_pretrained(
    VERBALIZER_MODEL
)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

verbalizer_model = AutoModelForSeq2SeqLM.from_pretrained(
    VERBALIZER_MODEL
).to(device)

In [ ]:
# 5.4 Define VectorToTextVerbilizer class

import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

class VectorToTextVerbalizer(nn.Module):
    def __init__(self, t5_model_name="google/flan-t5-small", vector_dim=64):
        super().__init__()
        self.t5 = AutoModelForSeq2SeqLM.from_pretrained(t5_model_name)
        self.t5_hidden_dim = self.t5.config.d_model
        self.vector_projection = nn.Linear(vector_dim, self.t5_hidden_dim)

    def forward(self, input_ids, attention_mask, latent_vectors, labels=None):
        text_embeds = self.t5.shared(input_ids)
        projected_vector = self.vector_projection(latent_vectors).unsqueeze(1)

        # FIX: Concatenate the vector BEFORE the text embeddings
        inputs_embeds = torch.cat([projected_vector, text_embeds], dim=1)

        # FIX: Add the active attention mask (1) to the front
        vector_mask = torch.ones((attention_mask.shape[0], 1), device=attention_mask.device)
        extended_attention_mask = torch.cat([vector_mask, attention_mask], dim=1)

        outputs = self.t5(
            inputs_embeds=inputs_embeds,
            attention_mask=extended_attention_mask,
            labels=labels
        )
        return outputs

    def generate(self, input_ids, attention_mask, latent_vectors, **kwargs):
        text_embeds = self.t5.shared(input_ids)
        projected_vector = self.vector_projection(latent_vectors).unsqueeze(1)

        # FIX: Mirror the forward pass changes for generation
        inputs_embeds = torch.cat([projected_vector, text_embeds], dim=1)

        vector_mask = torch.ones((attention_mask.shape[0], 1), device=attention_mask.device)
        extended_attention_mask = torch.cat([vector_mask, attention_mask], dim=1)

        return self.t5.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=extended_attention_mask,
            **kwargs
        )

In [ ]:
# 5.5 - Define NLADataset and data loader

from torch.utils.data import Dataset, DataLoader

class NlaDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_input_len=64, max_target_len=64):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len
        # Richer prompt — gives the encoder more tokens to condition on
        self.prompt_text = (
            "Summarise the semantic theme of this activation vector in one sentence: "
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        latent_vector = self.df.iloc[idx]["pca_vector"]
        # FIX 1: use the clean summary target, not raw input text
        target_text = str(self.df.iloc[idx]["summary_target"])

        inputs = self.tokenizer(
            self.prompt_text,
            max_length=self.max_input_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = self.tokenizer(
            target_text,
            max_length=self.max_target_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        label_ids = labels["input_ids"].squeeze(0)
        label_ids[label_ids == self.tokenizer.pad_token_id] = -100

        latent_vector = torch.tensor(latent_vector, dtype=torch.float32)
        # FIX 2: L2-normalise the latent before projection (matches paper sec. Method)
        latent_vector = latent_vector / (latent_vector.norm() + 1e-8)

        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "latent_vectors": latent_vector,
            "labels": label_ids,
            # Store original text for qualitative inspection
            "original_text": str(self.df.iloc[idx]["input_text"])[:200],
        }

# Rebuild verbalizer_df with the new summary_target column
verbalizer_df = pd.DataFrame({
    'pca_vector': list(activation_latents),
    'input_text': df["input_text"],
    'summary_target': df["summary_target"],
})

print(f"Created verbalizer_df with {len(verbalizer_df)} samples.")

train_dataset = NlaDataset(verbalizer_df, verbalizer_tokenizer)
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)

In [ ]:
# 5.6 VERBALIZER TRAINING — with warmup scheduler

from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR, SequentialLR
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
verbalizer_model = VectorToTextVerbalizer(vector_dim=64).to(device)

optimizer = AdamW(verbalizer_model.parameters(), lr=1e-4)

num_epochs = 20
num_training_steps = num_epochs * len(train_dataloader)
num_warmup_steps = int(0.1 * num_training_steps)  # 10% of steps for warmup

# Warmup phase: lr goes from 0 → 1e-4 over warmup steps
warmup_scheduler = LinearLR(
    optimizer,
    start_factor=0.01,       # starts at 1% of base lr (near zero)
    end_factor=1.0,          # ramps up to full base lr
    total_iters=num_warmup_steps
)

# Decay phase: lr goes from 1e-4 → ~0 over remaining steps
decay_scheduler = LinearLR(
    optimizer,
    start_factor=1.0,
    end_factor=0.01,         # decays to 1% of base lr by end
    total_iters=num_training_steps - num_warmup_steps
)

# Chain them: warmup first, then decay
scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, decay_scheduler],
    milestones=[num_warmup_steps]
)

print(f"Training for {num_epochs} epochs | {num_training_steps} steps | {num_warmup_steps} warmup steps")
verbalizer_model.train()

for epoch in range(num_epochs):
    total_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch in progress_bar:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        latent_vectors = batch["latent_vectors"].to(device)
        labels = batch["labels"].to(device)

        outputs = verbalizer_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            latent_vectors=latent_vectors,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(verbalizer_model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "lr": f"{scheduler.get_last_lr()[0]:.2e}"
        })

    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

In [ ]:
# 5.7 SAVE VERBALIZER CHECKPOINT (preserve trained AV model)

#VERBALIZER_DIR = f"{PROJECT_DIR}/verbalizer_checkpoint.pt"
#torch.save(verbalizer_model.state_dict(), VERBALIZER_DIR)
#verbalizer_tokenizer.save_pretrained(f"{PROJECT_DIR}/verbalizer_tokenizer")

VERBALIZER_MODEL_PATH = os.path.join(
    RUN_DIR,
    "verbalizer_checkpoint.pt"
)

VERBALIZER_TOKENIZER_DIR = os.path.join(
    RUN_DIR,
    "verbalizer_tokenizer"
)

torch.save(
    verbalizer_model.state_dict(),
    VERBALIZER_MODEL_PATH
)

verbalizer_tokenizer.save_pretrained(
    VERBALIZER_TOKENIZER_DIR
)

print("Saved verbalizer.")

In [ ]:
# 5.8 TEST VERBALIZER ON TRAINING EXAMPLES (jus before we generate explanations)

verbalizer_model.eval()
sample_idx = 0

# Get a single item from the dataset
sample = train_dataset[sample_idx]

input_ids = sample["input_ids"].unsqueeze(0).to(device)
attention_mask = sample["attention_mask"].unsqueeze(0).to(device)
latent_vector = sample["latent_vectors"].unsqueeze(0).to(device)
target_label = sample["labels"]

with torch.no_grad():
    generated_output = verbalizer_model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        latent_vectors=latent_vector,
        max_new_tokens=50,
        no_repeat_ngram_size=2,
        repetition_penalty=2.0,
        do_sample=True,
        temperature=0.7
    )


generated_text = verbalizer_tokenizer.decode(generated_output[0], skip_special_tokens=True)

clean_target_label = torch.where(
    target_label != -100,
    target_label,
    verbalizer_tokenizer.pad_token_id
)

original_text = verbalizer_tokenizer.decode(clean_target_label, skip_special_tokens=True)

print("="*80)
print(f"ORIGINAL TEXT (Target):\n{original_text}\n")
print(f"GENERATED EXPLANATION:\n{generated_text}")

In [ ]:
# 5.9 GENERATE NATURAL LANGUAGE EXPLANATIONS (creating the main explanation corpus)
import torch

# Put the custom model in evaluation mode (turns off dropout, optimizes memory)
verbalizer_model.eval()

generated_explanations = []

# Define how many samples to process (e.g., 1000, or the full dataset size)
num_samples_to_generate = min(1000, len(train_dataset))

print(f"Generating explanations for {num_samples_to_generate} samples...")

# Use no_grad() to prevent memory leaks during inference
with torch.no_grad():
    for i in range(num_samples_to_generate):

        # Extract the raw dictionary from PyTorch Dataset
        sample = train_dataset[i]

        # Extract tensors, add a batch dimension of 1 using unsqueeze(0), and push to GPU
        # Shape becomes [1, seq_len] or [1, vector_dim]
        input_ids = sample["input_ids"].unsqueeze(0).to(device)
        attention_mask = sample["attention_mask"].unsqueeze(0).to(device)
        latent_vector = sample["latent_vectors"].unsqueeze(0).to(device)

        # Pass the tensors directly into custom generate method
        output = verbalizer_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            latent_vectors=latent_vector,
            max_new_tokens=50,
            no_repeat_ngram_size=2,
            repetition_penalty=2.0,
            do_sample=True,
            temperature=0.7
        )

        # Decode the output tokens back into readable text
        text = verbalizer_tokenizer.decode(
            output[0],
            skip_special_tokens=True
        )

        generated_explanations.append(text)

        # Print progress marker - so we know something is happening, it could be frustrating ;-)
        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1} / {num_samples_to_generate}...")

print("Explanation generation complete")

In [ ]:
# 5.10 RECONSTRUCTION & EVALUATION (connects the verbalizer outputs back to FVE)
# explanation text → SentenceTransformer → MLP reconstructor → ĥ_l
# Then compare ĥ_l with the original h_l to get FVE.
# plots and success/failure samples

from sentence_transformers import SentenceTransformer
import numpy as np

'''
# Load the same embedding model used to train mlp_model
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 1. Encode the generated explanations
print("Encoding generated explanations with SentenceTransformer...")
explanation_embeddings = embedding_model.encode(
    generated_explanations,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(f"Explanation embedding shape: {explanation_embeddings.shape}")

# Also encode the original texts (to measure the upper bound / baseline)
original_texts = [
    str(train_dataset[i]["original_text"])
    for i in range(len(generated_explanations))
]
original_embeddings = embedding_model.encode(
    original_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)
'''

# Encode explanations and original texts using Gemma (replaces MiniLM)
print("Encoding generated explanations with Gemma...")
explanation_embeddings = encode_text_with_gemma(generated_explanations)

print("Encoding original texts with Gemma...")
original_texts = [
    str(train_dataset[i]["original_text"])
    for i in range(len(generated_explanations))
]
original_embeddings = encode_text_with_gemma(original_texts)

print(f"Explanation embeddings: {explanation_embeddings.shape}")
print(f"Original embeddings:    {original_embeddings.shape}")


# Get the original Gemma activations for these samples
#    (the full 2304-d vectors, NOT the PCA-compressed ones)
original_activations = np.array(
    df["activation_vector"].tolist()[:len(generated_explanations)]
)  # shape: (N, 2304)

# Run reconstruction through the MLP/Linear baseline (whichever is better)
#mlp_model.eval()
linear_model.eval()
with torch.no_grad():
    # Reconstructions from generated explanation embeddings
    expl_embed_tensor = torch.tensor(explanation_embeddings, dtype=torch.float32)
    #reconstructed_from_explanation = mlp_model(expl_embed_tensor).numpy()
    reconstructed_from_explanation = linear_model(expl_embed_tensor).numpy()

    # Reconstructions from original text embeddings (upper-bound baseline)
    orig_embed_tensor = torch.tensor(original_embeddings, dtype=torch.float32)
    #reconstructed_from_original = mlp_model(orig_embed_tensor).numpy()
    reconstructed_from_original = linear_model(orig_embed_tensor).numpy()

# Compute FVE and cosine similarity
def compute_fve(original, reconstructed):
    residual_var = np.var(original - reconstructed)
    total_var = np.var(original)
    return 1.0 - residual_var / total_var

def compute_mean_cosine(original, reconstructed):
    # Row-wise cosine similarity
    norms_o = np.linalg.norm(original, axis=1, keepdims=True) + 1e-8
    norms_r = np.linalg.norm(reconstructed, axis=1, keepdims=True) + 1e-8
    cos = np.sum((original / norms_o) * (reconstructed / norms_r), axis=1)
    return float(np.mean(cos))

fve_nla = compute_fve(original_activations, reconstructed_from_explanation)
cos_nla = compute_mean_cosine(original_activations, reconstructed_from_explanation)

fve_baseline = compute_fve(original_activations, reconstructed_from_original)
cos_baseline = compute_mean_cosine(original_activations, reconstructed_from_original)

print("\n===== RECONSTRUCTION RESULTS =====")
print(f"NLA roundtrip FVE  (explanation → embed → MLP → h_l): {fve_nla:.4f}")
print(f"NLA roundtrip CosSim:                                  {cos_nla:.4f}")
print(f"")
print(f"Upper-bound FVE  (original text → embed → MLP → h_l): {fve_baseline:.4f}")
print(f"Upper-bound CosSim:                                    {cos_baseline:.4f}")
print(f"")
print(f"Fidelity ratio (NLA / upper-bound FVE): {fve_nla / (fve_baseline + 1e-8):.3f}")
print(
    "(A ratio > 0.5 suggests the explanation preserves most of the recoverable signal; "
    "< 0.3 suggests the verbalizer output is too generic.)"
)

# PER-SAMPLE SCORES + VISUALISATIONS

import matplotlib.pyplot as plt

# Per-sample cosine similarity
norms_o = np.linalg.norm(original_activations, axis=1, keepdims=True) + 1e-8
norms_r = np.linalg.norm(reconstructed_from_explanation, axis=1, keepdims=True) + 1e-8
per_sample_cos = np.sum(
    (original_activations / norms_o) * (reconstructed_from_explanation / norms_r),
    axis=1
)  # shape: (N,)

# Per-sample FVE: treat each sample as a scalar residual normalised by total variance
total_var = np.var(original_activations)
per_sample_fve = 1.0 - (
    (original_activations - reconstructed_from_explanation) ** 2
).mean(axis=1) / total_var  # shape: (N,)

# ── Histograms ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(per_sample_fve, bins=30, color="steelblue", edgecolor="white")
axes[0].axvline(np.mean(per_sample_fve), color="red", linestyle="--", label=f"mean={np.mean(per_sample_fve):.3f}")
axes[0].set_title("FVE distribution (NLA roundtrip)")
axes[0].set_xlabel("Per-sample FVE")
axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].hist(per_sample_cos, bins=30, color="darkorange", edgecolor="white")
axes[1].axvline(np.mean(per_sample_cos), color="red", linestyle="--", label=f"mean={np.mean(per_sample_cos):.3f}")
axes[1].set_title("Cosine similarity distribution (NLA roundtrip)")
axes[1].set_xlabel("Per-sample cosine similarity")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
image_path_nla = os.path.join(
    RUN_DIR,
    "nla_distributions.png"
)
plt.savefig(image_path_nla, dpi=150)
#plt.savefig(f"{PROJECT_DIR}/nla_distributions.png", dpi=150)
plt.show()

# ── Top 10 successes and failures by cosine similarity ────────────────────────
ranked = np.argsort(per_sample_cos)
top10_fail_idx = ranked[:10]       # lowest cosine — worst reconstructions
top10_succ_idx = ranked[-10:][::-1]  # highest cosine — best reconstructions

def print_ranked(indices, label):
    print(f"\n{'='*80}")
    print(f"{label}")
    print(f"{'='*80}")
    for rank, idx in enumerate(indices):
        print(f"\n--- Rank {rank+1} | cosine={per_sample_cos[idx]:.4f} | fve={per_sample_fve[idx]:.4f} ---")
        print(f"ORIGINAL : {original_texts[idx][:150]}")
        print(f"EXPLANATION: {generated_explanations[idx][:150]}")

print_ranked(top10_succ_idx, "TOP 10 SUCCESSES (highest cosine similarity)")
print_ranked(top10_fail_idx, "TOP 10 FAILURES  (lowest cosine similarity)")

In [ ]:
# 5.11 INSPECT GENERATED EXPLANATIONS (Qualitative Evaluation)

print("--- QUALITATIVE INSPECTION ---\n")

for i in range(min(10, len(generated_explanations))):
    print("=" * 80)

    # 1. Original raw text — what the activation came from
    sample = train_dataset[i]
    original_text = sample["original_text"]       # Added in fixed NlaDataset
    print("ORIGINAL TEXT (source of activation):")
    print(original_text[:200])

    # 2. The training target — what the model was supervised to predict
    target_tokens = sample["labels"]
    clean_tokens = torch.where(
        target_tokens != -100,
        target_tokens,
        verbalizer_tokenizer.pad_token_id
    )
    summary_target = verbalizer_tokenizer.decode(clean_tokens, skip_special_tokens=True)
    print("\nSUMMARY TARGET (supervised signal):")
    print(summary_target)

    # 3. Generated explanation — what the verbalizer actually produced
    print("\nGENERATED EXPLANATION (verbalizer output):")
    print(generated_explanations[i])

print("=" * 80)

# **Section 6: Domain Comparisons (General & Software Code)**

In [ ]:
# 6.1 LETS RUN ENTIRE NLA PIPELINE ON CODE ACTIVATIONS (same thing we did for OpenWebText)

# Colelct code data sets
from datasets import load_dataset

NUM_CODE_SAMPLES = NUM_SAMPLES_TO_PROCESS  # match earlier OpenWebText sample count

print(f"Loading bigcode/the-stack-smol (Python only, streaming)...")
code_dataset = load_dataset(
    "bigcode/the-stack-smol",
    data_dir="data/python",
    streaming=True,
    split="train"
)
python_iter = iter(code_dataset)

code_records = []
collected = 0

while collected < NUM_CODE_SAMPLES:
    try:
        document = next(python_iter)
        raw_code = document["content"]

        if len(raw_code.strip()) < 50:   # skip trivially short files
            continue

        truncated = raw_code[:TEXT_TRUNCATION_LIMIT]
        inputs = tokenizer(
            truncated,
            return_tensors="pt",
            truncation=True
        ).to(model.device)

        final_token_position = int(inputs.input_ids.shape[1] - 1)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        final_token_vector = outputs.hidden_states[TARGET_LAYER][0, -1, :]
        activation_array = final_token_vector.detach().cpu().numpy().tolist()

        code_records.append({
            "sample_index": collected,
            "input_text": truncated,
            "activation_vector": activation_array,
            "language": "python"
        })

        collected += 1

        if collected % 100 == 0:
            print(f"Collected {collected}/{NUM_CODE_SAMPLES} code samples...")

    except StopIteration:
        print("Reached end of code dataset.")
        break
    except Exception as e:
        print(f"Skipped sample: {e}")
        continue

code_df = pd.DataFrame(code_records)
#code_df.to_parquet(os.path.join(LOCAL_REPO_PATH, "data/code_activations.parquet"))
code_df.to_parquet(
    os.path.join(
        RUN_DIR,
        "code_activations.parquet"
    )
)
print(f"Saved {len(code_df)} code activation samples.")

In [ ]:
# FURTHER PROCESS THE CODE DATA SET

# 1. PCA compress (reuse the SAME fitted pca object from OpenWebText)
#    Using the same PCA means both domains share the same latent space
code_activations_raw = np.array(code_df["activation_vector"].tolist())
code_activation_latents = pca.transform(code_activations_raw)  # NOT fit_transform

# 2. Build summary targets for code
def make_code_summary_target(code: str, max_chars: int = 200) -> str:
    """Extract the first meaningful comment or function signature as a summary."""
    lines = code.strip().split('\n')
    for line in lines:
        stripped = line.strip()
        # Prefer docstrings or comments
        if stripped.startswith('"""') or stripped.startswith("'''"):
            return stripped[:max_chars]
        if stripped.startswith('#') and len(stripped) > 10:
            return stripped[:max_chars]
        # Fall back to first function/class definition
        if stripped.startswith('def ') or stripped.startswith('class '):
            return stripped[:max_chars]
    return lines[0][:max_chars]  # fallback: first line

code_df["summary_target"] = code_df["input_text"].apply(make_code_summary_target)

# 3. Build dataset and generate explanations using the SAME trained verbalizer
code_verbalizer_df = pd.DataFrame({
    'pca_vector': list(code_activation_latents),
    'input_text': code_df["input_text"],
    'summary_target': code_df["summary_target"],
})

code_dataset_nla = NlaDataset(code_verbalizer_df, verbalizer_tokenizer)

verbalizer_model.eval()
code_explanations = []

print(f"Generating explanations for {len(code_dataset_nla)} code samples...")

with torch.no_grad():
    for i in range(len(code_dataset_nla)):
        sample = code_dataset_nla[i]

        input_ids = sample["input_ids"].unsqueeze(0).to(device)
        attention_mask = sample["attention_mask"].unsqueeze(0).to(device)
        latent_vector = sample["latent_vectors"].unsqueeze(0).to(device)

        output = verbalizer_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            latent_vectors=latent_vector,
            max_new_tokens=50,
            no_repeat_ngram_size=2,
            repetition_penalty=2.0,
            do_sample=True,
            temperature=0.7
        )

        text = verbalizer_tokenizer.decode(output[0], skip_special_tokens=True)
        code_explanations.append(text)

        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}/{len(code_dataset_nla)}...")

print("Code explanation generation complete!")

In [ ]:
# COMPARE OPENWEBTEXT vs CODE

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

'''
# 1. Encode code explanations and original code texts
print("Encoding code explanations...")
code_expl_embeddings = embedding_model.encode(
    code_explanations,
    show_progress_bar=True,
    convert_to_numpy=True
)

code_original_texts = code_df["input_text"].tolist()
code_orig_embeddings = embedding_model.encode(
    code_original_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)
'''

# Encode explanations and original texts using Gemma (replaces MiniLM)
print("Encoding generated explanations with Gemma...")
code_expl_embeddings = encode_text_with_gemma(code_explanations)

print("Encoding original texts with Gemma...")
code_original_texts = code_df["input_text"].tolist()
code_orig_embeddings = encode_text_with_gemma(code_original_texts)

print(f"Explanation embeddings: {code_expl_embeddings.shape}")
print(f"Original embeddings:    {code_orig_embeddings.shape}")


# 2. Reconstruct using the linear model
linear_model.eval()
with torch.no_grad():
    # Code: NLA roundtrip
    code_expl_tensor = torch.tensor(code_expl_embeddings, dtype=torch.float32)
    code_reconstructed_nla = linear_model(code_expl_tensor).numpy()

    # Code: upper bound
    code_orig_tensor = torch.tensor(code_orig_embeddings, dtype=torch.float32)
    code_reconstructed_upper = linear_model(code_orig_tensor).numpy()

# 3. Per-sample scores for code
total_var_code = np.var(code_activations_raw)

norms_co = np.linalg.norm(code_activations_raw, axis=1, keepdims=True) + 1e-8
norms_cr = np.linalg.norm(code_reconstructed_nla, axis=1, keepdims=True) + 1e-8
code_per_sample_cos = np.sum(
    (code_activations_raw / norms_co) * (code_reconstructed_nla / norms_cr),
    axis=1
)
code_per_sample_fve = 1.0 - (
    (code_activations_raw - code_reconstructed_nla) ** 2
).mean(axis=1) / total_var_code

# 4. Aggregate metrics
def aggregate(cos_arr, fve_arr, label):
    print(f"\n{label}")
    print(f"  FVE  — mean: {np.mean(fve_arr):.4f} | median: {np.median(fve_arr):.4f} | std: {np.std(fve_arr):.4f}")
    print(f"  CosSim — mean: {np.mean(cos_arr):.4f} | median: {np.median(cos_arr):.4f} | std: {np.std(cos_arr):.4f}")

print("\n===== DOMAIN COMPARISON =====")
aggregate(per_sample_cos, per_sample_fve, "OpenWebText (natural language)")
aggregate(code_per_sample_cos, code_per_sample_fve, "GitHub Python (code)")

# 5. Side-by-side distribution plots
fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 2, hspace=0.4, wspace=0.35)

# FVE distributions
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(per_sample_fve, bins=30, alpha=0.7, color="steelblue",
         edgecolor="white", label="OpenWebText")
ax1.hist(code_per_sample_fve, bins=30, alpha=0.7, color="darkorange",
         edgecolor="white", label="Python code")
ax1.axvline(np.mean(per_sample_fve), color="steelblue",
            linestyle="--", linewidth=1.5)
ax1.axvline(np.mean(code_per_sample_fve), color="darkorange",
            linestyle="--", linewidth=1.5)
ax1.set_title("FVE distribution by domain")
ax1.set_xlabel("Per-sample FVE")
ax1.set_ylabel("Count")
ax1.legend()

# Cosine distributions
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(per_sample_cos, bins=30, alpha=0.7, color="steelblue",
         edgecolor="white", label="OpenWebText")
ax2.hist(code_per_sample_cos, bins=30, alpha=0.7, color="darkorange",
         edgecolor="white", label="Python code")
ax2.axvline(np.mean(per_sample_cos), color="steelblue",
            linestyle="--", linewidth=1.5)
ax2.axvline(np.mean(code_per_sample_cos), color="darkorange",
            linestyle="--", linewidth=1.5)
ax2.set_title("Cosine similarity distribution by domain")
ax2.set_xlabel("Per-sample cosine similarity")
ax2.set_ylabel("Count")
ax2.legend()

# Box plots for direct comparison
ax3 = fig.add_subplot(gs[1, 0])
ax3.boxplot(
    [per_sample_fve, code_per_sample_fve],
    labels=["OpenWebText", "Python code"],
    patch_artist=True,
    boxprops=dict(facecolor="lightblue"),
    medianprops=dict(color="navy", linewidth=2)
)
ax3.set_title("FVE box plot")
ax3.set_ylabel("FVE")

ax4 = fig.add_subplot(gs[1, 1])
ax4.boxplot(
    [per_sample_cos, code_per_sample_cos],
    labels=["OpenWebText", "Python code"],
    patch_artist=True,
    boxprops=dict(facecolor="moccasin"),
    medianprops=dict(color="darkorange", linewidth=2)
)
ax4.set_title("Cosine similarity box plot")
ax4.set_ylabel("Cosine similarity")

plt.suptitle("NLA Reconstruction Quality: OpenWebText vs Python Code", fontsize=13)
image_path_domain = os.path.join(
    RUN_DIR,
    "domain_comparison.png"
)
plt.savefig(image_path_domain, dpi=150, bbox_inches="tight")
#plt.savefig(f"{PROJECT_DIR}/domain_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# 6. Qualitative inspection of code — top 5 successes and failures
code_ranked = np.argsort(code_per_sample_cos)
print("\n===== CODE: TOP 5 SUCCESSES =====")
for rank, idx in enumerate(code_ranked[-5:][::-1]):
    print(f"\n--- Rank {rank+1} | cos={code_per_sample_cos[idx]:.4f} ---")
    print(f"CODE    : {code_original_texts[idx][:150]}")
    print(f"EXPLAIN : {code_explanations[idx][:150]}")

print("\n===== CODE: TOP 5 FAILURES =====")
for rank, idx in enumerate(code_ranked[:5]):
    print(f"\n--- Rank {rank+1} | cos={code_per_sample_cos[idx]:.4f} ---")
    print(f"CODE    : {code_original_texts[idx][:150]}")
    print(f"EXPLAIN : {code_explanations[idx][:150]}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# INSPECT GENERATED EXPLANATIONS — CODE DOMAIN
# ─────────────────────────────────────────────────────────────────────────────

print("--- QUALITATIVE INSPECTION: PYTHON CODE ---\n")

for i in range(min(10, len(code_explanations))):
    print("=" * 80)

    sample = code_dataset_nla[i]
    original_code = sample["original_text"]
    print("ORIGINAL CODE (source of activation):")
    print(original_code[:200])

    target_tokens = sample["labels"]
    clean_tokens = torch.where(
        target_tokens != -100,
        target_tokens,
        verbalizer_tokenizer.pad_token_id
    )
    summary_target = verbalizer_tokenizer.decode(clean_tokens, skip_special_tokens=True)
    print("\nSUMMARY TARGET (supervised signal):")
    print(summary_target)

    print("\nGENERATED EXPLANATION (verbalizer output):")
    print(code_explanations[i])

print("=" * 80)

# **Section 7: SAVE/SYNC PROGRESS WITH REPOSITORY**

In [ ]:
# 7.1 Save Progress to Hithub


GH_TOKEN = userdata.get('GITHUB_TOKEN')
GH_USER = "olawoye"

gh_token = userdata.get('GITHUB_TOKEN')
domain = "github.com"
repo_url = f"https://{gh_token}@{domain}/{GH_USER}/{REPO_NAME}.git"

# Configure Git Identity
!git config --global user.email "ifeoluwa.olawoye@gmail.com"
!git config --global user.name "Ifeoluwa Olawoye"

!git lfs install
!git lfs track "*.parquet"

# Pull latest remote changes
%cd /content/{REPO_NAME}
!git pull origin main

# Add files, commit, and push (we exclude fiant files)
!git add .
!git commit -m "NLA project auto-sync checkpoints and updates"
!git push {repo_url} main

print("\n nla project progress synced with GitHub.")


In [ ]:
# 7.2 Explicitly track and add all model artifacts

import os
import shutil


# 1. Define paths
DRIVE_ROOT = f"{RUN_DIR}"
LOCAL_REPO_ROOT = f"/content/{REPO_NAME}" # Assuming your .git folder is here
#ARTEFACTS_DIR = os.path.join(LOCAL_REPO_ROOT, "artefacts")
ARTEFACTS_DIR = os.path.join(
    LOCAL_REPO_ROOT,
    "artefacts",
    CHECKPOINT_PATH
)

# 2. Create the folder locally in the repo
os.makedirs(ARTEFACTS_DIR, exist_ok=True)

# 3. Copy files from Drive to the local /artefacts/ folder
# We use shutil.copy2 to preserve file metadata
files_to_sync = [
    ("checkpoints", "dir"),
    ("verbalizer_checkpoint", "dir"),
    ("verbalizer_tokenizer", "dir"),
    ("domain_comparison.png", "file"),
    ("nla_distributions.png", "file"),
    ("experiment_info.json", "file"),
    ("requirements.txt", "file"),
    ("verbalizer_checkpoint.pt", "file")
]

for name, type in files_to_sync:
    src = os.path.join(DRIVE_ROOT, name)
    dst = os.path.join(ARTEFACTS_DIR, name)
    if type == "dir":
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)

# 4. Now sync this folder to GitHub
os.chdir(LOCAL_REPO_ROOT)
!git lfs track "artefacts/*.pt"
!git lfs track "artefacts/*.parquet"
!git lfs track "artefacts/*.safetensors"

!git add artefacts/
!git commit -m "Syncing artifacts to repository"
!git push origin main